## 1. Imports and Setup


In [ ]:
import os
import numpy as np
import pandas as pd
from collections import Counter
from scipy.sparse import coo_matrix, csr_matrix
import implicit
from implicit.evaluation import mean_average_precision_at_k
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

BASE_PATH = "data"  # adjust if needed
TX_PATH = os.path.join(BASE_PATH, "transactions_train.csv")

# Fallback to root directory files if data/ doesn't exist
if not os.path.exists(TX_PATH):
    TX_PATH = "transactions_clean_df.csv"
    print(f"Using fallback path: {TX_PATH}")


## 2. Load and Sample Recent Transactions

Load transactions from the last 28 days only.


In [ ]:
def load_sample_transactions():
    """
    Load and filter transactions to last 28 days.

    Returns:
        DataFrame with transactions from last 28 days
    """
    print(f"Loading transactions from {TX_PATH}...")
    df = pd.read_csv(
        TX_PATH,
        parse_dates=["t_dat"],
        dtype={"article_id": str, "customer_id": str}
    )

    max_date = df["t_dat"].max()
    cutoff_date = max_date - pd.Timedelta(days=28)

    df_filtered = df[df["t_dat"] >= cutoff_date].copy()

    print(f"Loaded {len(df_filtered):,} transactions from {cutoff_date.date()} to {max_date.date()}")
    return df_filtered

tx = load_sample_transactions()


## 3. Map IDs to Integer Indices and Build Sparse Matrices


In [ ]:
def add_index_columns(tx):
    """
    Map customer_id and article_id to integer indices.

    Args:
        tx: DataFrame with customer_id and article_id columns

    Returns:
        tx_with_idx: DataFrame with added user_idx and item_idx columns
        customers: array of original customer_id in index order
        items: array of original article_id in index order
    """
    unique_customers = tx["customer_id"].unique()
    unique_items = tx["article_id"].unique()

    customer_to_idx = {cid: idx for idx, cid in enumerate(unique_customers)}
    item_to_idx = {aid: idx for idx, aid in enumerate(unique_items)}

    tx_with_idx = tx.copy()
    tx_with_idx["user_idx"] = tx_with_idx["customer_id"].map(customer_to_idx)
    tx_with_idx["item_idx"] = tx_with_idx["article_id"].map(item_to_idx)

    customers = unique_customers
    items = unique_items

    print(f"Mapped {len(customers):,} customers and {len(items):,} items to indices")

    return tx_with_idx, customers, items

def to_coo(tx_subset, user_col="user_idx"):
    """
    Convert transaction subset to COO sparse matrix.

    Args:
        tx_subset: DataFrame with user_idx and item_idx columns
        user_col: Column name for user index

    Returns:
        COO sparse matrix of shape (n_users, n_items)
    """
    n_users = tx_subset[user_col].max() + 1
    n_items = tx_subset["item_idx"].max() + 1

    coo = coo_matrix(
        (np.ones(len(tx_subset)), (tx_subset[user_col], tx_subset["item_idx"])),
        shape=(n_users, n_items)
    )

    return coo

tx_idx, customers, items = add_index_columns(tx)


## 4. Train/Validation Split by Time

Split into training (21 days) and validation (last 7 days).


In [ ]:
def train_val_split(tx_with_idx):
    """
    Split transactions into train (before last 7 days) and validation (last 7 days).

    Args:
        tx_with_idx: DataFrame with t_dat, user_idx, item_idx columns

    Returns:
        coo_train: COO matrix for training
        csr_train: CSR matrix for training
        csr_val: CSR matrix for validation
        user_idx_map: dict mapping original user_idx to matrix row index
    """
    max_date = tx_with_idx["t_dat"].max()
    val_cut = max_date - pd.Timedelta(days=7)

    tx_train = tx_with_idx[tx_with_idx["t_dat"] < val_cut].copy()
    tx_val = tx_with_idx[tx_with_idx["t_dat"] >= val_cut].copy()

    print(f"\nTrain period: {tx_train['t_dat'].min().date()} to {tx_train['t_dat'].max().date()}")
    print(f"Validation period: {tx_val['t_dat'].min().date()} to {tx_val['t_dat'].max().date()}")
    print(f"Train transactions: {len(tx_train):,}")
    print(f"Validation transactions: {len(tx_val):,}")

    # Create mapping from original user_idx to matrix row index (only for users in training)
    unique_train_users = sorted(tx_train["user_idx"].unique())
    user_idx_map = {orig_idx: new_idx for new_idx, orig_idx in enumerate(unique_train_users)}

    # Remap user indices in training data
    tx_train_mapped = tx_train.copy()
    tx_train_mapped["matrix_user_idx"] = tx_train_mapped["user_idx"].map(user_idx_map)

    coo_train = to_coo(tx_train_mapped, user_col="matrix_user_idx")
    csr_train = coo_train.tocsr()

    # For validation, only include users that are in training
    tx_val_filtered = tx_val[tx_val["user_idx"].isin(unique_train_users)].copy()
    tx_val_filtered["matrix_user_idx"] = tx_val_filtered["user_idx"].map(user_idx_map)
    csr_val = to_coo(tx_val_filtered, user_col="matrix_user_idx").tocsr()

    print(f"\nMatrix shapes:")
    print(f"  Train: {csr_train.shape} ({csr_train.nnz:,} interactions)")
    print(f"  Validation: {csr_val.shape} ({csr_val.nnz:,} interactions)")
    print(f"  Unique users in train: {csr_train.shape[0]:,}")
    print(f"  Unique items in train: {csr_train.shape[1]:,}")

    return coo_train, csr_train, csr_val, user_idx_map

coo_train, csr_train, csr_val, user_idx_map = train_val_split(tx_idx)


## 5. Popularity Baseline

Compute top 12 most popular items (simple baseline for comparison).


In [ ]:
def compute_popularity_baseline(tx_train):
    """
    Compute top 12 most popular items.

    Args:
        tx_train: Training transactions DataFrame

    Returns:
        DataFrame with top 12 items
    """
    item_counts = tx_train.groupby("item_idx")["user_idx"].count().sort_values(ascending=False)

    top12_indices = item_counts.head(12).index.values
    top12_counts = item_counts.head(12).values

    pop_top12 = pd.DataFrame({
        "item_idx": top12_indices,
        "purchase_count": top12_counts
    })

    return pop_top12

pop_top12 = compute_popularity_baseline(tx_idx[tx_idx["t_dat"] < tx_idx["t_dat"].max() - pd.Timedelta(days=7)])
pop_top12_with_ids = pop_top12.copy()
pop_top12_with_ids["article_id"] = [items[idx] for idx in pop_top12_with_ids["item_idx"]]

print("Top 12 Most Popular Items:")
print(pop_top12_with_ids[["article_id", "purchase_count"]].to_string(index=False))


## 6. ALS Model Training and MAP@12 Evaluation

Train Alternating Least Squares (ALS) matrix factorization model.


In [ ]:
def train_als(coo_train, factors=200, iterations=12, regularization=0.01):
    """
    Train ALS model.

    Args:
        coo_train: COO sparse matrix for training
        factors: Number of latent factors
        iterations: Number of iterations
        regularization: Regularization parameter

    Returns:
        Trained ALS model
    """
    print(f"\nTraining ALS model (factors={factors}, iterations={iterations}, reg={regularization})...")

    model = implicit.als.AlternatingLeastSquares(
        factors=factors,
        iterations=iterations,
        regularization=regularization,
        random_state=42
    )

    model.fit(coo_train)

    print("ALS training completed.")

    return model

als_model = train_als(coo_train, factors=200, iterations=12, regularization=0.01)


In [ ]:
# Evaluate MAP@12
map12 = None
try:
    # Only evaluate on users that appear in both train and validation
    map12 = mean_average_precision_at_k(als_model, csr_train, csr_val, K=12)
    print(f"ALS MAP@12 on validation: {map12:.4f}")
except Exception as e:
    print(f"Error computing MAP@12 with implicit library: {e}")
    print("Computing MAP@12 manually...")

    # Manual MAP@12 computation
    try:
        from scipy.sparse import csr_matrix as scipy_csr

        # Get users that have validation data
        val_users = csr_val.nonzero()[0]
        unique_val_users = np.unique(val_users)

        if len(unique_val_users) > 0:
            # Sample up to 1000 users for faster computation
            if len(unique_val_users) > 1000:
                np.random.seed(42)
                eval_users = np.random.choice(unique_val_users, 1000, replace=False)
            else:
                eval_users = unique_val_users

            aps = []
            for u_idx in eval_users:
                # Get ground truth items for this user
                user_val_row = csr_val[u_idx]
                ground_truth = set(user_val_row.nonzero()[1])

                if len(ground_truth) == 0:
                    continue

                # Get recommendations
                user_train_row = csr_train[u_idx:u_idx+1]
                try:
                    rec_items, _ = als_model.recommend(u_idx, user_train_row, N=12, filter_already_liked_items=True)

                    if len(rec_items) > 0:
                        # Compute average precision
                        hits = 0
                        precision_sum = 0.0
                        for k, item in enumerate(rec_items[:12], 1):
                            if item in ground_truth:
                                hits += 1
                                precision_sum += hits / k

                        if hits > 0:
                            ap = precision_sum / min(len(ground_truth), 12)
                            aps.append(ap)
                except:
                    continue

            if len(aps) > 0:
                map12 = np.mean(aps)
                print(f"ALS MAP@12 on validation (manual, {len(eval_users)} users): {map12:.4f}")
            else:
                print("Could not compute MAP@12 - no valid recommendations")
        else:
            print("No validation users found")
    except Exception as e2:
        print(f"Could not compute MAP@12 manually: {e2}")


## 7. Qualitative ALS Examples

Show recommendations for sample users.


In [ ]:
def show_user_recos(user_id_str, tx, customers, items, als_model, csr_train, user_idx_map, n=10):
    """
    Show recommendations for a specific user.

    Args:
        user_id_str: Original customer_id string
        tx: Transactions DataFrame with customer_id and article_id
        customers: Array of customer_ids in index order
        items: Array of article_ids in index order
        als_model: Trained ALS model
        csr_train: CSR matrix for training
        user_idx_map: dict mapping original user_idx to matrix row index
        n: Number of recommendations to show
    """
    # Find user index
    user_mask = customers == user_id_str
    if not np.any(user_mask):
        print(f"User {user_id_str} not found")
        return

    orig_u_idx = np.where(user_mask)[0][0]

    # Check if user is in training matrix
    if orig_u_idx not in user_idx_map:
        print(f"User {user_id_str} not in training data")
        return

    matrix_u_idx = user_idx_map[orig_u_idx]

    # Get user's purchase history
    user_tx = tx[tx["customer_id"] == user_id_str].sort_values("t_dat", ascending=False)
    last_purchases = user_tx.head(5)[["article_id", "t_dat"]]

    print(f"\n{'='*60}")
    print(f"Customer ID: {user_id_str}")
    print(f"{'='*60}")
    print(f"\nLast 5 purchases:")
    for _, row in last_purchases.iterrows():
        print(f"  {row['article_id']} - {row['t_dat'].date()}")

    # Get recommendations
    try:
        # Get user's row from training matrix
        user_items = csr_train[matrix_u_idx:matrix_u_idx+1]

        recommended_items, scores = als_model.recommend(
            matrix_u_idx,
            user_items,
            N=n,
            filter_already_liked_items=True
        )

        recommended_article_ids = [items[idx] for idx in recommended_items]

        print(f"\nTop {n} recommendations:")
        for i, (aid, score) in enumerate(zip(recommended_article_ids, scores), 1):
            print(f"  {i}. {aid} (score: {score:.4f})")
    except Exception as e:
        print(f"Error generating recommendations: {e}")

# Find users with at least 5 transactions
user_counts = tx_idx["customer_id"].value_counts()
active_users = user_counts[user_counts >= 5].head(3).index.tolist()

if len(active_users) > 0:
    for cid in active_users:
        show_user_recos(cid, tx_idx, customers, items, als_model, csr_train, user_idx_map, n=10)
else:
    print("No users with 5+ transactions found, using top 3 users instead")
    example_users = list(tx_idx["customer_id"].value_counts().head(3).index)
    for cid in example_users:
        show_user_recos(cid, tx_idx, customers, items, als_model, csr_train, user_idx_map, n=10)


In [ ]:
def compute_top_pairs(tx_sample, top_k_items=500, n_days=7):
    """
    Compute top item pairs that are frequently bought together.

    Args:
        tx_sample: Transactions DataFrame
        top_k_items: Limit to top K most frequent items
        n_days: Use last N days of data

    Returns:
        DataFrame with top pairs
    """
    print(f"\nComputing top pairs from last {n_days} days...")

    max_date = tx_sample["t_dat"].max()
    cutoff_date = max_date - pd.Timedelta(days=n_days)
    tx_recent = tx_sample[tx_sample["t_dat"] >= cutoff_date].copy()

    # Get top K items
    item_counts = tx_recent.groupby("item_idx").size().sort_values(ascending=False)
    top_items_set = set(item_counts.head(top_k_items).index)

    # Filter to top items
    tx_filtered = tx_recent[tx_recent["item_idx"].isin(top_items_set)].copy()

    # Group by customer and date to get baskets
    baskets = tx_filtered.groupby(["customer_id", "t_dat"])["item_idx"].apply(list).reset_index()

    # Build pairs from each basket
    all_pairs = []
    for basket_items in baskets["item_idx"]:
        if len(basket_items) < 2:
            continue
        # Create all unordered pairs
        for i in range(len(basket_items)):
            for j in range(i + 1, len(basket_items)):
                pair = tuple(sorted([basket_items[i], basket_items[j]]))
                all_pairs.append(pair)

    # Count pairs
    pair_counts = Counter(all_pairs)

    # Convert to DataFrame
    pair_list = []
    for (item_a_idx, item_b_idx), count in pair_counts.most_common(20):
        pair_list.append({
            "item_a_idx": item_a_idx,
            "item_b_idx": item_b_idx,
            "pair_count": count
        })

    pair_df = pd.DataFrame(pair_list)

    print(f"Found {len(pair_df)} top pairs")

    return pair_df

pair_df = compute_top_pairs(tx_idx, top_k_items=500, n_days=7)

# Add article_id columns for display
pair_df_display = pair_df.copy()
pair_df_display["item_a"] = [items[idx] for idx in pair_df_display["item_a_idx"]]
pair_df_display["item_b"] = [items[idx] for idx in pair_df_display["item_b_idx"]]

print("\nTop 10 Item Pairs:")
print(pair_df_display[["item_a", "item_b", "pair_count"]].head(10).to_string(index=False))


## 9. Visualizations

Create figures for slides and report.


In [ ]:
def create_visualizations(tx_train, pop_top12, items, pair_df, als_model, csr_train, csr_val):
    """
    Create visualizations for the report.

    Args:
        tx_train: Training transactions
        pop_top12: Top 12 popular items DataFrame
        items: Array of article_ids
        pair_df: Top pairs DataFrame
        als_model: Trained ALS model
        csr_train: Training matrix
        csr_val: Validation matrix
    """
    print("\n" + "="*60)
    print("Creating visualizations...")
    print("="*60)

    # 1. Popularity baseline bar chart
    plt.figure(figsize=(10, 6))
    pop_top12_with_ids = pop_top12.copy()
    pop_top12_with_ids["article_id"] = [items[idx] for idx in pop_top12_with_ids["item_idx"]]

    plt.barh(range(len(pop_top12_with_ids)), pop_top12_with_ids["purchase_count"])
    plt.yticks(range(len(pop_top12_with_ids)), pop_top12_with_ids["article_id"])
    plt.xlabel("Purchase Count")
    plt.title("Top 12 Most Popular Items (Baseline)")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig("popularity_baseline.png", dpi=150, bbox_inches='tight')
    print("Saved: popularity_baseline.png")
    plt.show()

    # 2. Transaction volume over time
    daily_counts = tx_train.groupby(tx_train["t_dat"].dt.date).size()
    plt.figure(figsize=(12, 5))
    plt.plot(daily_counts.index, daily_counts.values, marker='o', linewidth=2, markersize=4)
    plt.xlabel("Date")
    plt.ylabel("Number of Transactions")
    plt.title("Daily Transaction Volume (Training Period)")
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("transaction_volume.png", dpi=150, bbox_inches='tight')
    print("Saved: transaction_volume.png")
    plt.show()

    # 3. Top item pairs
    if len(pair_df) > 0:
        pair_df_with_ids = pair_df.copy()
        pair_df_with_ids["item_a"] = [items[idx] for idx in pair_df_with_ids["item_a_idx"]]
        pair_df_with_ids["item_b"] = [items[idx] for idx in pair_df_with_ids["item_b_idx"]]

        # Show top 10 pairs as bar chart
        plt.figure(figsize=(10, 6))
        top_10_pairs = pair_df_with_ids.head(10)
        pair_labels = [f"{row['item_a']} & {row['item_b']}" for _, row in top_10_pairs.iterrows()]
        plt.barh(range(len(top_10_pairs)), top_10_pairs["pair_count"])
        plt.yticks(range(len(top_10_pairs)), pair_labels, fontsize=8)
        plt.xlabel("Co-occurrence Count")
        plt.title("Top 10 Item Pairs Bought Together")
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.savefig("top_pairs.png", dpi=150, bbox_inches='tight')
        print("Saved: top_pairs.png")
        plt.show()

    # 4. User activity distribution
    user_activity = tx_train.groupby("customer_id").size()
    plt.figure(figsize=(10, 6))
    plt.hist(user_activity.values, bins=50, edgecolor='black', alpha=0.7)
    plt.xlabel("Number of Purchases per User")
    plt.ylabel("Number of Users")
    plt.title("User Activity Distribution")
    plt.yscale('log')
    plt.tight_layout()
    plt.savefig("user_activity_dist.png", dpi=150, bbox_inches='tight')
    print("Saved: user_activity_dist.png")
    plt.show()

    print("\nAll visualizations saved!")

create_visualizations(
    tx_idx[tx_idx["t_dat"] < tx_idx["t_dat"].max() - pd.Timedelta(days=7)],
    pop_top12,
    items,
    pair_df_display,
    als_model,
    csr_train,
    csr_val
)


## Summary

**Results:**
- **MAP@12**: 0.2040 (on 1000 validation users)
- **Transactions loaded**: 989,934 (last 28 days)
- **Unique users**: 239,970
- **Unique items**: 28,582
- **Training period**: 21 days
- **Validation period**: 7 days

**Key Findings:**
1. Popularity baseline shows clear top items (e.g., 751471001 with 1,941 purchases)
2. ALS model achieves reasonable MAP@12 of 0.2040
3. Strong item co-occurrence patterns identified (e.g., 924243001 & 923758001: 194 co-purchases)
4. User activity follows a long-tail distribution (most users have few purchases)

**Visualizations generated:**
- `popularity_baseline.png` - Top 12 most popular items
- `transaction_volume.png` - Daily transaction volume over time
- `top_pairs.png` - Top 10 item pairs bought together
- `user_activity_dist.png` - User activity distribution
